In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


INPUT_FILE = "V4 Usage Training.csv"
OUTPUT_FOLDER = Path("forecast_results")

# Rolling backtest settings
VALIDATION_START = "2026-01-01"
FORECAST_MONTHS = 6

TARGET_COLUMN = "Monthly Inventory Issues"
PART_COLUMN = "fpartno"
DATE_COLUMN = "Date"

# Version 4 MPL feature
MPL_COLUMN = "MPL Qty"

LAGS = [1, 2, 3, 6, 12]

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

print("Version 4 settings loaded.")


In [ ]:
data = pd.read_csv(INPUT_FILE)

required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    MPL_COLUMN,
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

data[DATE_COLUMN] = pd.to_datetime(
    data[DATE_COLUMN],
    errors="raise",
)

# Keep only data from 2023 onward
data = data[
    data[DATE_COLUMN] >= pd.Timestamp("2023-01-01")
].copy()

data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN],
    errors="raise",
)

# Convert MPL quantity to numeric
data[MPL_COLUMN] = pd.to_numeric(
    data[MPL_COLUMN],
    errors="coerce",
)

# Blank MPL means no MPL activity
data[MPL_COLUMN] = (
    data[MPL_COLUMN]
    .fillna(0)
)

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)

if duplicates.any():

    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows were found:\n"
        f"{duplicate_rows.head(20)}"
    )

print(f"Rows loaded: {len(data):,}")

print(
    f"Unique parts: "
    f"{data[PART_COLUMN].nunique():,}"
)

print(
    f"First date: "
    f"{data[DATE_COLUMN].min()}"
)

print(
    f"Last date: "
    f"{data[DATE_COLUMN].max()}"
)

print(
    f"Total MPL Qty: "
    f"{data[MPL_COLUMN].sum():,.0f}"
)

print(
    f"Rows with MPL activity: "
    f"{(data[MPL_COLUMN] > 0).sum():,}"
)


In [ ]:
def create_training_features(historical_data):

    feature_data = historical_data.copy()

    # =====================================================
    # DATE / TIME FEATURES
    # =====================================================

    feature_data["month"] = (
        feature_data[DATE_COLUMN].dt.month
    )

    feature_data["year"] = (
        feature_data[DATE_COLUMN].dt.year
    )

    feature_data["quarter"] = (
        feature_data[DATE_COLUMN].dt.quarter
    )

    feature_data["time_idx"] = (
        np.arange(len(feature_data))
    )


    # =====================================================
    # USAGE LAG FEATURES
    # =====================================================

    for lag in LAGS:

        feature_data[f"lag_{lag}"] = (
            feature_data[TARGET_COLUMN].shift(lag)
        )


    # Shift usage so the current month's actual usage
    # is never used to predict itself.
    prior_usage = (
        feature_data[TARGET_COLUMN].shift(1)
    )


    # =====================================================
    # ROLLING USAGE AVERAGES
    # =====================================================

    feature_data["rolling_mean_3"] = (
        prior_usage.rolling(3).mean()
    )

    feature_data["rolling_mean_6"] = (
        prior_usage.rolling(6).mean()
    )

    feature_data["rolling_mean_12"] = (
        prior_usage.rolling(12).mean()
    )


    # =====================================================
    # ROLLING USAGE MEDIANS
    # =====================================================

    feature_data["rolling_median_3"] = (
        prior_usage.rolling(3).median()
    )

    feature_data["rolling_median_6"] = (
        prior_usage.rolling(6).median()
    )

    feature_data["rolling_median_12"] = (
        prior_usage.rolling(12).median()
    )


    # =====================================================
    # ANNUAL USAGE
    # =====================================================

    feature_data["rolling_total_12"] = (
        prior_usage.rolling(12).sum()
    )


    # =====================================================
    # USAGE VARIABILITY
    # =====================================================

    feature_data["rolling_std_3"] = (
        prior_usage.rolling(3).std()
    )

    feature_data["rolling_std_6"] = (
        prior_usage.rolling(6).std()
    )

    feature_data["rolling_std_12"] = (
        prior_usage.rolling(12).std()
    )


    # =====================================================
    # USAGE RANGE
    # =====================================================

    feature_data["rolling_min_12"] = (
        prior_usage.rolling(12).min()
    )

    feature_data["rolling_max_12"] = (
        prior_usage.rolling(12).max()
    )


    # =====================================================
    # USAGE TRENDS
    # =====================================================

    feature_data["trend_3"] = (
        feature_data["lag_1"]
        - feature_data["lag_3"]
    )

    feature_data["trend_6"] = (
        feature_data["lag_1"]
        - feature_data["lag_6"]
    )


    # =====================================================
    # ZERO-USAGE BEHAVIOR
    # =====================================================

    feature_data[
        "zero_month_percentage_12"
    ] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values: (
                values == 0
            ).mean(),
            raw=True,
        )
    )


    # =====================================================
    # COEFFICIENT OF VARIATION
    # =====================================================

    feature_data[
        "coefficient_variation_12"
    ] = (
        feature_data["rolling_std_12"]
        /
        feature_data[
            "rolling_mean_12"
        ].replace(
            0,
            np.nan,
        )
    )


    # =====================================================
    # RECENT VS ANNUAL USAGE
    # =====================================================

    feature_data["recent_vs_annual"] = (
        feature_data["rolling_mean_3"]
        -
        feature_data["rolling_mean_12"]
    )


    # =====================================================
    # MPL FEATURES
    # =====================================================

    # Shift MPL so the model never sees MPL
    # from the month being predicted.
    prior_mpl = (
        feature_data[MPL_COLUMN].shift(1)
    )


    # MPL lag features
    feature_data["mpl_lag_1"] = (
        feature_data[MPL_COLUMN].shift(1)
    )

    feature_data["mpl_lag_3"] = (
        feature_data[MPL_COLUMN].shift(3)
    )

    feature_data["mpl_lag_6"] = (
        feature_data[MPL_COLUMN].shift(6)
    )


    # MPL totals
    feature_data["mpl_rolling_sum_6"] = (
        prior_mpl.rolling(6).sum()
    )

    feature_data["mpl_rolling_sum_12"] = (
        prior_mpl.rolling(12).sum()
    )


    # Number of months with MPL activity
    feature_data["mpl_months_12"] = (
        prior_mpl
        .rolling(12)
        .apply(
            lambda values: (
                values > 0
            ).sum(),
            raw=True,
        )
    )


    # Percentage of months with MPL activity
    feature_data["mpl_percentage_12"] = (
        prior_mpl
        .rolling(12)
        .apply(
            lambda values: (
                values > 0
            ).mean(),
            raw=True,
        )
    )


    # Potential demand =
    # actual usage + missing-parts demand
    #
    # This is only a feature.
    # Actual usage remains the prediction target.
    feature_data["potential_demand_12"] = (
        feature_data["rolling_total_12"]
        +
        feature_data["mpl_rolling_sum_12"]
    )


    # =====================================================
    # FEATURE LIST
    # =====================================================

    feature_columns = [

        # Date / time
        "month",
        "year",
        "quarter",
        "time_idx",

        # Usage lags
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",

        # Rolling averages
        "rolling_mean_3",
        "rolling_mean_6",
        "rolling_mean_12",

        # Rolling medians
        "rolling_median_3",
        "rolling_median_6",
        "rolling_median_12",

        # Annual usage
        "rolling_total_12",

        # Variability
        "rolling_std_3",
        "rolling_std_6",
        "rolling_std_12",

        # Range
        "rolling_min_12",
        "rolling_max_12",

        # Demand behavior
        "zero_month_percentage_12",
        "coefficient_variation_12",
        "recent_vs_annual",

        # Trends
        "trend_3",
        "trend_6",

        # MPL
        "mpl_lag_1",
        "mpl_lag_3",
        "mpl_lag_6",
        "mpl_rolling_sum_6",
        "mpl_rolling_sum_12",
        "mpl_months_12",
        "mpl_percentage_12",
        "potential_demand_12",
    ]


    training_rows = (
        feature_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=feature_columns
        )
        .reset_index(drop=True)
    )


    return (
        training_rows,
        feature_columns,
    )


print("Version 4 MPL feature function created.")


In [ ]:
def train_model(training_rows, feature_columns):

    model = LGBMRegressor(
        objective="poisson",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        min_child_samples=10,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        training_rows[feature_columns],
        training_rows[TARGET_COLUMN],
    )

    return model


print("Training function created.")


In [ ]:
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
                MPL_COLUMN,
            ]
        ]
        .copy()
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    predictions = []


    for _ in range(forecast_months):

        next_date = (
            forecast_history[DATE_COLUMN].max()
            + pd.DateOffset(months=1)
        )

        recent_usage = (
            forecast_history[TARGET_COLUMN]
        )

        recent_mpl = (
            forecast_history[MPL_COLUMN]
        )


        # =====================================================
        # DATE / TIME
        # =====================================================

        future_row = {
            "month": next_date.month,
            "year": next_date.year,
            "quarter": next_date.quarter,
            "time_idx": len(forecast_history),
        }


        # =====================================================
        # USAGE FEATURES
        # =====================================================

        for lag in LAGS:

            future_row[f"lag_{lag}"] = (
                recent_usage.iloc[-lag]
            )


        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]


        # Rolling averages
        future_row["rolling_mean_3"] = (
            last_3.mean()
        )

        future_row["rolling_mean_6"] = (
            last_6.mean()
        )

        future_row["rolling_mean_12"] = (
            last_12.mean()
        )


        # Rolling medians
        future_row["rolling_median_3"] = (
            last_3.median()
        )

        future_row["rolling_median_6"] = (
            last_6.median()
        )

        future_row["rolling_median_12"] = (
            last_12.median()
        )


        # Annual usage
        future_row["rolling_total_12"] = (
            last_12.sum()
        )


        # Standard deviations
        future_row["rolling_std_3"] = (
            last_3.std()
        )

        future_row["rolling_std_6"] = (
            last_6.std()
        )

        future_row["rolling_std_12"] = (
            last_12.std()
        )


        # Usage range
        future_row["rolling_min_12"] = (
            last_12.min()
        )

        future_row["rolling_max_12"] = (
            last_12.max()
        )


        # Zero-month percentage
        future_row[
            "zero_month_percentage_12"
        ] = (
            last_12.eq(0).mean()
        )


        # Coefficient of variation
        rolling_mean_12 = (
            future_row["rolling_mean_12"]
        )

        rolling_std_12 = (
            future_row["rolling_std_12"]
        )

        if rolling_mean_12 != 0:

            future_row[
                "coefficient_variation_12"
            ] = (
                rolling_std_12
                / rolling_mean_12
            )

        else:

            future_row[
                "coefficient_variation_12"
            ] = 0.0


        # Recent versus annual usage
        future_row["recent_vs_annual"] = (
            future_row["rolling_mean_3"]
            -
            future_row["rolling_mean_12"]
        )


        # Trends
        future_row["trend_3"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-3]
        )

        future_row["trend_6"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-6]
        )


        # =====================================================
        # MPL FEATURES
        # =====================================================

        last_mpl_6 = (
            recent_mpl.iloc[-6:]
        )

        last_mpl_12 = (
            recent_mpl.iloc[-12:]
        )


        # MPL lags
        future_row["mpl_lag_1"] = (
            recent_mpl.iloc[-1]
        )

        future_row["mpl_lag_3"] = (
            recent_mpl.iloc[-3]
        )

        future_row["mpl_lag_6"] = (
            recent_mpl.iloc[-6]
        )


        # MPL totals
        future_row["mpl_rolling_sum_6"] = (
            last_mpl_6.sum()
        )

        future_row["mpl_rolling_sum_12"] = (
            last_mpl_12.sum()
        )


        # Number of months with MPL activity
        future_row["mpl_months_12"] = (
            (last_mpl_12 > 0).sum()
        )


        # Percentage of months with MPL activity
        future_row["mpl_percentage_12"] = (
            (last_mpl_12 > 0).mean()
        )


        # Potential demand
        future_row["potential_demand_12"] = (
            future_row["rolling_total_12"]
            +
            future_row["mpl_rolling_sum_12"]
        )


        # =====================================================
        # MODEL INPUT
        # =====================================================

        future_features = pd.DataFrame(
            [future_row],
            columns=feature_columns,
        )


        # =====================================================
        # PREDICTION
        # =====================================================

        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )

        predicted_usage = max(
            0.0,
            predicted_usage,
        )


        predictions.append(
            {
                PART_COLUMN: part_number,
                DATE_COLUMN: next_date,
                "Predicted Usage": predicted_usage,
            }
        )


        # =====================================================
        # UPDATE HISTORY
        # =====================================================

        # Cell 6 only forecasts one month at a time
        # during the rolling backtest.
        #
        # Therefore this future MPL placeholder does
        # not affect the rolling validation results.

        new_history_row = pd.DataFrame(
            {
                DATE_COLUMN: [next_date],
                TARGET_COLUMN: [
                    predicted_usage
                ],
                MPL_COLUMN: [0],
            }
        )


        forecast_history = pd.concat(
            [
                forecast_history,
                new_history_row,
            ],
            ignore_index=True,
        )


    return pd.DataFrame(predictions)


print("Version 4 MPL forecast function created.")


In [ ]:
all_results = []
errors = []

validation_start = pd.Timestamp(
    VALIDATION_START
)

validation_months = FORECAST_MONTHS


for part_number in sorted(
    data[PART_COLUMN].dropna().unique()
):

    print(f"\nProcessing: {part_number}")

    try:

        part_data = (
            data[
                data[PART_COLUMN] == part_number
            ]
            .copy()
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )

        part_results = []


        for month_number in range(
            validation_months
        ):

            prediction_date = (
                validation_start
                + pd.DateOffset(
                    months=month_number
                )
            )


            # Use all actual usage and MPL
            # information available before
            # the prediction month.
            historical_data = part_data[
                part_data[DATE_COLUMN]
                < prediction_date
            ].copy()


            actual_row = part_data[
                part_data[DATE_COLUMN]
                == prediction_date
            ][
                [
                    DATE_COLUMN,
                    TARGET_COLUMN,
                ]
            ].copy()


            if actual_row.empty:

                raise ValueError(
                    f"No actual usage found for "
                    f"{prediction_date:%Y-%m}"
                )


            training_rows, feature_columns = (
                create_training_features(
                    historical_data
                )
            )


            if training_rows.empty:

                raise ValueError(
                    f"No usable training rows for "
                    f"{prediction_date:%Y-%m}"
                )


            # Retrain using everything available
            # before this prediction month.
            model = train_model(
                training_rows,
                feature_columns,
            )


            # Forecast one month ahead.
            one_month_forecast = (
                forecast_future_months(
                    model=model,
                    historical_data=historical_data,
                    feature_columns=feature_columns,
                    forecast_months=1,
                    part_number=part_number,
                )
            )


            predicted_usage = (
                one_month_forecast[
                    "Predicted Usage"
                ].iloc[0]
            )


            actual_usage = (
                actual_row[
                    TARGET_COLUMN
                ].iloc[0]
            )


            error = (
                predicted_usage
                - actual_usage
            )


            part_results.append(
                {
                    PART_COLUMN: part_number,
                    DATE_COLUMN: prediction_date,
                    "Predicted Usage": (
                        predicted_usage
                    ),
                    "Actual Usage": (
                        actual_usage
                    ),
                    "Error": error,
                    "Absolute Error": abs(
                        error
                    ),
                    "Training Through": (
                        historical_data[
                            DATE_COLUMN
                        ].max()
                    ),
                }
            )


        all_results.append(
            pd.DataFrame(
                part_results
            )
        )

        print(
            f"Finished: {part_number}"
        )


    except Exception as error:

        errors.append(
            {
                PART_COLUMN: str(
                    part_number
                ),
                "Error": str(error),
            }
        )

        print(
            f"Skipped {part_number}: "
            f"{error}"
        )


if not all_results:

    raise RuntimeError(
        "No rolling forecasts "
        "completed successfully."
    )


print(
    "\nVersion 4 rolling backtest complete."
)


In [ ]:
results = (
    pd.concat(
        all_results,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)


# Keep decimal predictions for accuracy calculations.
# Create rounded values only for presentation.
results["Predicted Usage Rounded"] = (
    results["Predicted Usage"]
    .round()
    .astype(int)
)

results["Actual Usage Rounded"] = (
    results["Actual Usage"]
    .round()
    .astype(int)
)


display(
    results[
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Training Through",
            "Predicted Usage Rounded",
            "Actual Usage Rounded",
            "Error",
            "Absolute Error",
        ]
    ]
)


In [ ]:
profile_data = data[
    data[DATE_COLUMN]
    < pd.Timestamp(VALIDATION_START)
].copy()


demand_profile = (
    profile_data
    .groupby(PART_COLUMN)
    .agg(

        Average_Monthly_Usage=(
            TARGET_COLUMN,
            "mean",
        ),

        Median_Monthly_Usage=(
            TARGET_COLUMN,
            "median",
        ),

        Zero_Month_Percentage=(
            TARGET_COLUMN,
            lambda x: (
                x == 0
            ).mean(),
        ),

        Nonzero_Months=(
            TARGET_COLUMN,
            lambda x: (
                x > 0
            ).sum(),
        ),

        Average_MPL_Qty=(
            MPL_COLUMN,
            "mean",
        ),

        Total_MPL_Qty=(
            MPL_COLUMN,
            "sum",
        ),

        Months_With_MPL=(
            MPL_COLUMN,
            lambda x: (
                x > 0
            ).sum(),
        ),

        MPL_Month_Percentage=(
            MPL_COLUMN,
            lambda x: (
                x > 0
            ).mean(),
        ),
    )
    .reset_index()
)


demand_profile[
    "Zero_Month_Percentage"
] *= 100

demand_profile[
    "MPL_Month_Percentage"
] *= 100


# Round for easier viewing
demand_profile[
    "Average_Monthly_Usage"
] = (
    demand_profile[
        "Average_Monthly_Usage"
    ].round(2)
)

demand_profile[
    "Median_Monthly_Usage"
] = (
    demand_profile[
        "Median_Monthly_Usage"
    ].round(2)
)

demand_profile[
    "Average_MPL_Qty"
] = (
    demand_profile[
        "Average_MPL_Qty"
    ].round(2)
)

demand_profile[
    "Zero_Month_Percentage"
] = (
    demand_profile[
        "Zero_Month_Percentage"
    ].round(1)
)

demand_profile[
    "MPL_Month_Percentage"
] = (
    demand_profile[
        "MPL_Month_Percentage"
    ].round(1)
)


display(demand_profile)


In [ ]:
import boto3
from sagemaker_studio import Project
import io


# Get project S3 path
proj = Project()
project_s3_root = proj.s3.root

s3_parts = (
    project_s3_root
    .replace("s3://", "")
    .split("/", 1)
)

bucket = s3_parts[0]

prefix = (
    s3_parts[1]
    if len(s3_parts) > 1
    else ""
)

s3 = boto3.client("s3")


# Select columns to upload
results_to_upload = results[
    [
        PART_COLUMN,
        DATE_COLUMN,
        "Training Through",
        "Predicted Usage",
        "Predicted Usage Rounded",
        "Actual Usage",
        "Error",
        "Absolute Error",
    ]
].copy()


# Create CSV in memory
csv_buffer = io.StringIO()

results_to_upload.to_csv(
    csv_buffer,
    index=False,
)


# Version 4 gets its own output file
s3_key = (
    f"{prefix}/results/"
    "version_4_mpl_rolling_backtest_results.csv"
)


# Upload
s3.put_object(
    Bucket=bucket,
    Key=s3_key,
    Body=csv_buffer.getvalue().encode(
        "utf-8"
    ),
    ContentType="text/csv",
)


s3_path = (
    f"s3://{bucket}/{s3_key}"
)


print(
    "Successfully uploaded Version 4 "
    "MPL rolling backtest results to S3!"
)

print(
    f"S3 path: {s3_path}"
)

print(
    f"Rows uploaded: "
    f"{len(results_to_upload)}"
)
